# Stage 0 — Project setup and system config

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import os

def find_project_root(start: Path) -> Path:
    current = start.resolve()

    while current != current.parent:
        if (current / "src").exists() and (current / "artifacts").exists():
            return current
        if (current / "src").exists():
            return current
        current = current.parent

    raise RuntimeError("Could not locate project root containing src/")


PROJECT_ROOT = find_project_root(Path.cwd())

SRC = PROJECT_ROOT / "src"
ARTIFACTS = PROJECT_ROOT / "artifacts"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC:", SRC)
print("ARTIFACTS:", ARTIFACTS)


#-------------------
# Change working directory to project root
os.chdir(PROJECT_ROOT)
print("CWD:", Path.cwd())

In [ ]:
SYSTEM_NAME = "spp"
PROFILE_NAME = "spp_swarm_spread"
TRIAL = 1

SYSTEM_ORDER = 2
INTERACTION_TYPE = "energy"

TRAIN_SPLIT = "train"
TRAIN_FIT_SPLIT = "train_fit"
TEST_SPLIT = "test"
KDE_SPLIT = "kde"

TRAIN_DATASET_TEMPLATE = "dataset_train_trial_{trial:03d}"
TRAIN_FIT_DATASET_TEMPLATE = "dataset_train_fit_trial_{trial:03d}"

TRAIN_DATASET_NAME = TRAIN_DATASET_TEMPLATE.format(
    trial=TRIAL,
)

TRAIN_FIT_DATASET_NAME = TRAIN_FIT_DATASET_TEMPLATE.format(
    trial=TRIAL,
)

TEST_DATASET_NAME = "dataset_test"
KDE_DATASET_NAME = "dataset_kde"


FIT_END_TIME = 4
PREDICTION_END_TIME = 10

ANIMATION_END_TIME = 20
ANI_PROFILE_NAME = "spp_swarm"
TEST_ANIMATION_DATASET_NAME = f"{TEST_DATASET_NAME}_animation_T{ANIMATION_END_TIME:g}_N{ANIMATION_N}"
ANIMATION_NUM_FRAMES = 2001
ANIMATION_N = 100

In [3]:
SYSTEM_ARTIFACT_DIR = ARTIFACTS / SYSTEM_NAME / PROFILE_NAME

DATASET_ROOT = ARTIFACTS / "datasets"
DATASET_DIR = DATASET_ROOT / SYSTEM_NAME / PROFILE_NAME
ANIMATION_DATASET_DIR = DATASET_ROOT / SYSTEM_NAME / ANI_PROFILE_NAME
FEATURE_ROOT = ARTIFACTS / "features"
FEATURE_DIR = FEATURE_ROOT / SYSTEM_NAME / PROFILE_NAME
BASIS_ROOT = ARTIFACTS / "bases"
BASIS_DIR = BASIS_ROOT / SYSTEM_NAME / PROFILE_NAME
FIT_ROOT = ARTIFACTS / "fits"
FIT_DIR = FIT_ROOT / SYSTEM_NAME / PROFILE_NAME
ROLLOUT_ROOT = ARTIFACTS / "rollouts"
ROLLOUT_DIR = ROLLOUT_ROOT / SYSTEM_NAME / PROFILE_NAME
EVALUATION_ROOT = ARTIFACTS / "evaluations"
EVALUATION_DIR = EVALUATION_ROOT / SYSTEM_NAME / PROFILE_NAME
DENSITY_ROOT = ARTIFACTS / "densities"
FIGURE_DIR = SYSTEM_ARTIFACT_DIR / "figures"

print("System:", SYSTEM_NAME)
print("Profile:", PROFILE_NAME)
print("Trial:", TRIAL)
print("System order:", SYSTEM_ORDER)
print("Interaction:", INTERACTION_TYPE)

print("Train dataset:", TRAIN_DATASET_NAME)
print("Train-fit dataset:", TRAIN_FIT_DATASET_NAME)
print("Test dataset:", TEST_DATASET_NAME)
print("KDE dataset:", KDE_DATASET_NAME)

print("FIT_END_TIME:", FIT_END_TIME)
print("PREDICTION_END_TIME:", PREDICTION_END_TIME)

System: spp
Profile: spp_swarm_spread
Trial: 1
System order: 2
Interaction: energy
Train dataset: dataset_train_trial_001
Train-fit dataset: dataset_train_fit_trial_001
Test dataset: dataset_test
KDE dataset: dataset_kde
FIT_END_TIME: 4
PREDICTION_END_TIME: 10


In [4]:
from ips_learning.data.dataset_repository import DatasetRepository

from ips_learning.features.feature_builder import FeatureBuilder
from ips_learning.features.feature_repository import FeatureRepository

from ips_learning.bases.specs import (
    BasisConfig,
    RadialBasisSpec,
    EnvironmentBasisSpec,
)
from ips_learning.bases.builders import BasisBuilder
from ips_learning.bases.repository import BasisRepository

from ips_learning.fitting.assembler import NormalEquationAssembler
from ips_learning.fitting.solver import LinearSystemSolver
from ips_learning.fitting.system_repository import NormalSystemRepository

from ips_learning.models.factories import LearnedModelFactory

from ips_learning.rollout.simulator import LearnedSystemRollout
from ips_learning.rollout.scenarios import RolloutScenarioBuilder
from ips_learning.rollout.repository import RolloutRepository

from ips_learning.evaluation.run_evaluation import EvaluationRunner

from ips_learning.statistics.density import (
    KernelDensityEstimator,
    EnvironmentDensityEstimator,
)

from ips_learning.statistics.density_repository import DensityRepository

In [5]:
# General helper functions

def ensure_dir(path: Path) -> Path:
    """
    Create directory if missing and return the Path.
    """
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def path_exists(path: Path) -> bool:
    """
    Check whether a file or directory exists.
    """
    return Path(path).exists()


def load_json(path: Path) -> dict:
    """
    Load JSON file.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: dict, path: Path) -> None:
    """
    Save JSON file with parent directory creation.
    """
    path = Path(path)
    ensure_dir(path.parent)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, sort_keys=True)


def first_existing_path(*paths: Path) -> Path | None:
    """
    Return first existing path from candidates.
    """
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def pretty_path(path: Path) -> Path:
    """
    Show project-relative path when possible.
    """
    path = Path(path).resolve()

    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return path

In [6]:
# Repository objects

dataset_repo = DatasetRepository(root_dir=DATASET_ROOT)
feature_repo = FeatureRepository(root_dir=FEATURE_ROOT)
basis_repo = BasisRepository(root_dir=BASIS_ROOT)
fit_repo = NormalSystemRepository(root_dir=FIT_ROOT)
rollout_repo = RolloutRepository(root_dir=ROLLOUT_ROOT)
density_repo = DensityRepository(root=DENSITY_ROOT)

# Stage 1: Dataset preparation

In [7]:
def dataset_artifact_complete(artifact_dir):
    return (
        (artifact_dir / "trajectories.npz").is_file()
        and (artifact_dir / "metadata.json").is_file()
    )


expected_dataset_dirs = {
    "train": DATASET_DIR / TRAIN_SPLIT / TRAIN_DATASET_NAME,
    "train_fit": DATASET_DIR / TRAIN_FIT_SPLIT / TRAIN_FIT_DATASET_NAME,
    "test": DATASET_DIR / TEST_SPLIT / TEST_DATASET_NAME,
    "kde": DATASET_DIR / KDE_SPLIT / KDE_DATASET_NAME,
}

for name, artifact_dir in expected_dataset_dirs.items():
    status = "FOUND" if dataset_artifact_complete(artifact_dir) else "MISSING"
    print(f"{name:10s}", status, pretty_path(artifact_dir))


train      FOUND artifacts/datasets/spp/spp_swarm_spread/train/dataset_train_trial_001
train_fit  FOUND artifacts/datasets/spp/spp_swarm_spread/train_fit/dataset_train_fit_trial_001
test       FOUND artifacts/datasets/spp/spp_swarm_spread/test/dataset_test
kde        FOUND artifacts/datasets/spp/spp_swarm_spread/kde/dataset_kde


In [8]:
from ips_learning.experiments.generate_spp_data import build_spp_dataset_profile
from ips_learning.experiments.generate_dataset_collection import (
    generate_standard_dataset_collection,
)

profile = build_spp_dataset_profile(PROFILE_NAME)

required_standard_dataset_dirs = []

# All full training trials and their derived fitting windows.
for q in range(profile.train.num_trials):
    trial_id = q + 1

    train_name = profile.train.dataset_name_template.format(
        trial=trial_id,
    )
    required_standard_dataset_dirs.append(
        DATASET_DIR / profile.train.split / train_name
    )

    train_fit_name = TRAIN_FIT_DATASET_TEMPLATE.format(
        trial=trial_id,
    )
    required_standard_dataset_dirs.append(
        DATASET_DIR / TRAIN_FIT_SPLIT / train_fit_name
    )

# Shared reference/evaluation datasets.
required_standard_dataset_dirs.extend(
    [
        DATASET_DIR / profile.test.split / profile.test.dataset_name,
        DATASET_DIR / profile.kde.split / profile.kde.dataset_name,
        DATASET_DIR
        / profile.validation.split
        / profile.validation.dataset_name,
    ]
)

missing_standard_dataset_dirs = [
    artifact_dir
    for artifact_dir in required_standard_dataset_dirs
    if not dataset_artifact_complete(artifact_dir)
]

if not missing_standard_dataset_dirs:
    print("Complete standard dataset collection found. Skipping generation.")

else:
    print(
        f"{len(missing_standard_dataset_dirs)} dataset artifacts are missing "
        "or incomplete. Regenerating the standard collection."
    )

    outputs = generate_standard_dataset_collection(
        profile=profile,
        save_train_fit_split=True,
        fit_end_time=FIT_END_TIME,
        fit_split_name=TRAIN_FIT_SPLIT,
        fit_name_template=TRAIN_FIT_DATASET_TEMPLATE,
    )

    print("Generated dataset collection:")
    for key, value in outputs.items():
        print(key, ":", value)


Complete standard dataset collection found. Skipping generation.


In [9]:
# Load datasets
train_dataset = dataset_repo.load(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    split=TRAIN_SPLIT,
    dataset_name=TRAIN_DATASET_NAME,
)

train_fit_dataset = dataset_repo.load(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    split=TRAIN_FIT_SPLIT,
    dataset_name=TRAIN_FIT_DATASET_NAME,
)

test_dataset = dataset_repo.load(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    split=TEST_SPLIT,
    dataset_name=TEST_DATASET_NAME,
)

kde_dataset = dataset_repo.load(
    system_name=SYSTEM_NAME,
    profile_name=PROFILE_NAME,
    split=KDE_SPLIT,
    dataset_name=KDE_DATASET_NAME,
)

print("Datasets loaded.")

Datasets loaded.


In [10]:
# Animation dataset is generated later, only when the
# animation rollout is requested.

test_animation_dir = (
    ANIMATION_DATASET_DIR
    / TEST_SPLIT
    / TEST_ANIMATION_DATASET_NAME
)

animation_status = (
    "FOUND"
    if dataset_artifact_complete(test_animation_dir)
    else "NOT GENERATED"
)

print(
    "Animation dataset:",
    animation_status,
    pretty_path(test_animation_dir),
)


Animation dataset: NOT GENERATED artifacts/datasets/spp/spp_swarm/test/dataset_test_animation_T20


In [11]:
# Verify standard dataset shapes
def summarize_dataset(name, dataset):
    print(f"\n{name}")
    print("-" * len(name))
    print("t:  ", dataset.t.shape)
    print("X:  ", dataset.X.shape)
    print("V:  ", None if dataset.V is None else dataset.V.shape)
    print("LHS:", dataset.LHS.shape)
    print("U0: ", dataset.U0.shape)

    print("system_name:     ", dataset.metadata.get("system_name"))
    print("system_order:    ", dataset.metadata.get("system_order"))
    print("interaction_type:", dataset.metadata.get("interaction_type"))
    print("profile_name:    ", dataset.metadata.get("profile_name"))
    print("split:           ", dataset.metadata.get("split"))
    print("dataset_name:    ", dataset.metadata.get("dataset_name"))
    print("T:               ", dataset.metadata.get("T"))
    print("M:               ", dataset.metadata.get("M"))
    print("L:               ", dataset.metadata.get("L"))


summarize_dataset("train_dataset", train_dataset)
summarize_dataset("train_fit_dataset", train_fit_dataset)
summarize_dataset("test_dataset", test_dataset)
summarize_dataset("kde_dataset", kde_dataset)



train_dataset
-------------
t:   (1001,)
X:   (50, 1001, 10, 2)
V:   (50, 1001, 10, 2)
LHS: (50, 1001, 10, 2)
U0:  (50, 40)
system_name:      spp
system_order:     2
interaction_type: energy
profile_name:     spp_swarm_spread
split:            train
dataset_name:     dataset_train_trial_001
T:                10.0
M:                50
L:                1001

train_fit_dataset
-----------------
t:   (401,)
X:   (50, 401, 10, 2)
V:   (50, 401, 10, 2)
LHS: (50, 401, 10, 2)
U0:  (50, 40)
system_name:      spp
system_order:     2
interaction_type: energy
profile_name:     spp_swarm_spread
split:            train_fit
dataset_name:     dataset_train_fit_trial_001
T:                4.0
M:                50
L:                401

test_dataset
------------
t:   (1001,)
X:   (10, 1001, 10, 2)
V:   (10, 1001, 10, 2)
LHS: (10, 1001, 10, 2)
U0:  (10, 40)
system_name:      spp
system_order:     2
interaction_type: energy
profile_name:     spp_swarm_spread
split:            test
dataset_name:     data

In [12]:
# Validate metadata consistency
datasets = {
    "train": train_dataset,
    "train_fit": train_fit_dataset,
    "test": test_dataset,
    "kde": kde_dataset,
}

for name, dataset in datasets.items():
    assert dataset.metadata.get("system_name") == SYSTEM_NAME, (
        name,
        dataset.metadata.get("system_name"),
        SYSTEM_NAME,
    )

    assert dataset.metadata.get("system_order") == SYSTEM_ORDER, (
        name,
        dataset.metadata.get("system_order"),
        SYSTEM_ORDER,
    )

    assert dataset.metadata.get("interaction_type") == INTERACTION_TYPE, (
        name,
        dataset.metadata.get("interaction_type"),
        INTERACTION_TYPE,
    )

    assert dataset.X.shape == dataset.LHS.shape, (
        name,
        dataset.X.shape,
        dataset.LHS.shape,
    )

    if SYSTEM_ORDER == 1:
        assert dataset.V is None, name

    elif SYSTEM_ORDER == 2:
        assert dataset.V is not None, name
        assert dataset.V.shape == dataset.X.shape, (
            name,
            dataset.V.shape,
            dataset.X.shape,
        )

print("Dataset metadata consistency checks passed.")

Dataset metadata consistency checks passed.


# Satge 2


Cell — Feature artifact paths

Cell — Build/load features

Cell — Inspect/validate features

Cell — Stage 2.2 markdown

Cell — Basis artifact paths

Cell — Basis config

Cell — Build/load basis

Cell — Inspect/validate basis

Cell — KDE markdown

Cell — Load/compute KDE densities

## Stage 2.1: Feature construction

This stage converts saved trajectory data into pairwise geometric features needed for fitting.

Second-order systems:

\[
(X,V,LHS), \qquad LHS=\ddot X
\]

First-order systems:

\[
(X,LHS), \qquad LHS=\dot X
\]

Shared interaction features:

\[
R_D = X_j - X_i,
\qquad
R = \|X_j-X_i\|
\]

Second-order-only feature:

\[
VR_D = V_j - V_i.
\]

For first-order systems, `V=None` and `VR_D=None`.

In [13]:
FEATURE_NAME = TRAIN_FIT_DATASET_NAME

feature_artifact_dir = FEATURE_DIR / TRAIN_FIT_SPLIT / FEATURE_NAME

feature_npz_path = feature_artifact_dir / "features.npz"
feature_metadata_path = feature_artifact_dir / "metadata.json"

print("Feature artifact directory:")
print(pretty_path(feature_artifact_dir))

Feature artifact directory:
artifacts/features/spp/spp_swarm_spread/train_fit/dataset_train_fit_trial_001


In [14]:
if feature_npz_path.exists() and feature_metadata_path.exists():
    print("Loading existing feature artifact.")

    features = feature_repo.load(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=TRAIN_FIT_SPLIT,
        feature_name=FEATURE_NAME,
    )

else:
    print("Feature artifact not found. Building features from train_fit dataset.")

    features = FeatureBuilder().build(train_fit_dataset)

    feature_repo.save(
        features=features,
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=TRAIN_FIT_SPLIT,
        feature_name=FEATURE_NAME,
        overwrite=True,
    )

    print("Saved feature artifact.")

Loading existing feature artifact.


In [15]:
print("Feature shapes")
print("t:   ", features.t.shape)
print("X:   ", features.X.shape)
print("V:   ", None if features.V is None else features.V.shape)
print("LHS: ", features.LHS.shape)
print("R_D: ", features.R_D.shape)
print("R:   ", features.R.shape)

if getattr(features, "VR_D", None) is not None:
    print("VR_D:", features.VR_D.shape)
else:
    print("VR_D: None")

print("\nFeature metadata")
print("system_name:     ", features.metadata.get("system_name"))
print("system_order:    ", features.metadata.get("system_order"))
print("interaction_type:", features.metadata.get("interaction_type"))

assert features.metadata.get("system_name") == SYSTEM_NAME
assert features.metadata.get("system_order") == SYSTEM_ORDER
assert features.metadata.get("interaction_type") == INTERACTION_TYPE

assert features.X.shape == features.LHS.shape

if SYSTEM_ORDER == 1:
    assert features.V is None
    assert features.VR_D is None

elif SYSTEM_ORDER == 2:
    assert features.V is not None
    assert features.V.shape == features.X.shape

print("Feature metadata checks passed.")

Feature shapes
t:    (401,)
X:    (50, 401, 10, 2)
V:    (50, 401, 10, 2)
LHS:  (50, 401, 10, 2)
R_D:  (50, 401, 10, 10, 2)
R:    (50, 401, 10, 10, 1)
VR_D: (50, 401, 10, 10, 2)

Feature metadata
system_name:      spp
system_order:     2
interaction_type: energy
Feature metadata checks passed.


## Stage 2.2: Basis construction and localization

This stage builds localized basis functions for the two learned components:

\[
\theta_\phi \rightarrow \phi(r)
\]

\[
\theta_f \rightarrow f(z)
\]

where the environmental variable \(z\) depends on the system order:

Second-order:

\[
z=v, \qquad \ddot X=f(V)+f_\phi(X)
\]

First-order:

\[
z=x, \qquad \dot X=f(X)+f_\phi(X)
\]

The basis domain is localized from the feature artifact:

- `r_min`, `r_max` from observed pairwise geometry
- `env_min`, `env_max` from the environmental variable
- `environment_variable = "velocity"` for second-order
- `environment_variable = "position"` for first-order

In [16]:
BASIS_NAME = TRAIN_FIT_DATASET_NAME

basis_artifact_dir = BASIS_DIR / TRAIN_FIT_SPLIT / BASIS_NAME
basis_metadata_path = basis_artifact_dir / "basis_metadata.json"

print("Basis artifact directory:")
print(pretty_path(basis_artifact_dir))

Basis artifact directory:
artifacts/bases/spp/spp_swarm_spread/train_fit/dataset_train_fit_trial_001


In [17]:
# Define basis configuration
basis_config = BasisConfig(
    radial=RadialBasisSpec(
        family="bspline",
        degree=1,
        n_basis=10,
    ),
    environment=EnvironmentBasisSpec(
        family="bspline",
        degree=1,
        n_basis_per_dim=10,
    ),
)

In [18]:
from ips_learning.systems.spp import parametric_f_basis

basis_builder = BasisBuilder(basis_config)

if basis_metadata_path.exists():
    print("Loading existing basis artifact.")

    basis_bundle = basis_repo.load(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=TRAIN_FIT_SPLIT,
        basis_name=BASIS_NAME,
    )

    if basis_bundle.parametric_f_basis is None:
        print("Reattaching SPP parametric f basis.")
        basis_bundle.parametric_f_basis = parametric_f_basis()


else:
    print("Basis artifact not found. Building localized basis from features.")
    basis_bundle = basis_builder.build(
        features=features,
        parametric_f_basis=parametric_f_basis(),
    )

    basis_repo.save(
        basis_bundle=basis_bundle,
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=TRAIN_FIT_SPLIT,
        basis_name=BASIS_NAME,
        overwrite=True,
    )

    print("Saved basis artifact.")

print("Parametric f terms:", [t.name for t in basis_bundle.parametric_f_basis.terms])

Loading existing basis artifact.
Reattaching SPP parametric f basis.
Parametric f terms: ['v_1', 'v_2', '||v||^2 v_1', '||v||^2 v_2']


In [19]:
# Inspect localization metadata
loc = basis_bundle.localization_metadata

print("Basis localization metadata")
print("---------------------------")

for key, value in loc.items():
    print(f"{key}: {value}")

print("\nBasis sizes")
print("-----------")
print("radial n_basis:", basis_bundle.radial_basis.n_basis)

if hasattr(basis_bundle, "environment_basis") and basis_bundle.environment_basis is not None:
    print("environment n_basis_per_dim:", basis_bundle.environment_basis.n_basis_per_dim)
else:
    raise ValueError("Missing environment basis.")

assert "r_min" in loc
assert "r_max" in loc
assert loc["r_max"] > loc["r_min"]

assert "environment_variable" in loc
assert loc["environment_variable"] in {"velocity", "position"}

expected_env_variable = "position" if SYSTEM_ORDER == 1 else "velocity"
assert loc["environment_variable"] == expected_env_variable, (
    loc["environment_variable"],
    expected_env_variable,
)

assert "env_min" in loc
assert "env_max" in loc

env_min = np.asarray(loc["env_min"], dtype=float)
env_max = np.asarray(loc["env_max"], dtype=float)

assert np.all(env_max > env_min)

# Backward-compatible metadata may still exist
if "v_min" in loc and "v_max" in loc:
    v_min = np.asarray(loc["v_min"], dtype=float)
    v_max = np.asarray(loc["v_max"], dtype=float)
    assert np.all(v_max > v_min)

print("Basis localization checks passed.")

Basis localization metadata
---------------------------
alignment_basis_source: radial_basis
basis_name: dataset_train_fit_trial_001
basis_stage: 2.2
energy_basis_source: radial_basis
env_max: [3.5777087639996634, 3.5777087639996634]
env_min: [-3.5777087639996634, -3.5777087639996634]
environment_degree: 1
environment_family: bspline
environment_knots: [[-3.5777087639996634, -3.5777087639996634, -2.7826623719997383, -1.987615979999813, -1.1925695879998877, -0.39752319599996255, 0.39752319599996255, 1.192569587999888, 1.9876159799998132, 2.7826623719997383, 3.5777087639996634, 3.5777087639996634], [-3.5777087639996634, -3.5777087639996634, -2.7826623719997383, -1.987615979999813, -1.1925695879998877, -0.39752319599996255, 0.39752319599996255, 1.192569587999888, 1.9876159799998132, 2.7826623719997383, 3.5777087639996634, 3.5777087639996634]]
environment_variable: velocity
interaction_type: energy
n_alignment_basis: 10
n_energy_basis: 10
n_environment_basis_per_dim: 10
n_parametric_f_term

In [20]:
# Parametric f basis check
if getattr(basis_bundle, "parametric_f_basis", None) is None:
    raise ValueError(
        "Missing parametric_f_basis. "
        "For parametric_f runs, reattach it from the system module, e.g. "
        "`from ips_learning.systems.levi import parametric_f_basis`."
    )

parametric_terms = [term.name for term in basis_bundle.parametric_f_basis.terms]

print("parametric f terms:", parametric_terms)
print("n_parametric_f_terms:", basis_bundle.parametric_f_basis.n_terms)

assert basis_bundle.parametric_f_basis.n_terms > 0

if "parametric_f_terms" in loc:
    assert loc["parametric_f_terms"] == parametric_terms, (
        loc["parametric_f_terms"],
        parametric_terms,
    )

parametric f terms: ['v_1', 'v_2', '||v||^2 v_1', '||v||^2 v_2']
n_parametric_f_terms: 4


### Load or compute KDE/reference density artifacts

This computes reusable density artifacts once and saves them.

The kernel density is always 1D and is built from pairwise distances.

The environmental density depends on the system:

- second-order: density of \(V\)
- first-order: density of \(X\)

For Kuramoto, environmental density is 1D.
For Levi/Phototaxis, environmental density is 2D.



In [21]:
try:
    kernel_density_ref, env_density_ref = density_repo.load(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=KDE_SPLIT,
        name=KDE_DATASET_NAME,
    )
    print("Loaded existing KDE/reference density artifact.")

except FileNotFoundError:
    print("KDE/reference density artifact not found. Computing once.")

    kde_dataset = dataset_repo.load(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=KDE_SPLIT,
        dataset_name=KDE_DATASET_NAME,
    )

    kde_features = FeatureBuilder().build(kde_dataset)

    kernel_density_ref = KernelDensityEstimator().fit_from_R(
        kde_features.R,
        bandwidth=None,
        kernel="gaussian",
    )

    if SYSTEM_ORDER == 1:
        env_samples_source = kde_features.X
        environment_variable = "position"

    elif SYSTEM_ORDER == 2:
        env_samples_source = kde_features.V
        environment_variable = "velocity"

    else:
        raise ValueError(f"Unsupported SYSTEM_ORDER={SYSTEM_ORDER}")

    env_density_ref = EnvironmentDensityEstimator().fit_from_Z(
        env_samples_source,
        bandwidth=None,
        kernel="gaussian",
    )

    density_repo.save(
        dist_est=kernel_density_ref,
        vel_est=env_density_ref,   # keep repository arg name if not yet renamed
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        split=KDE_SPLIT,
        name=KDE_DATASET_NAME,
        metadata={
            "system_name": SYSTEM_NAME,
            "profile_name": PROFILE_NAME,
            "source_split": KDE_SPLIT,
            "source_dataset_name": KDE_DATASET_NAME,
            "artifact_type": "kde_reference_density",
            "system_order": SYSTEM_ORDER,
            "environment_variable": environment_variable,
        },
    )

    print("Saved KDE/reference density artifact.")

Loaded existing KDE/reference density artifact.


# Stage 3: Fitting

Cell — Fit artifact path

Cell — Solver + assemble/solve/save fit

Cell — Optional normal-system inspection

Cell — Fit diagnostics and consistency checks

This stage assembles and solves the normal equations

\[
A\theta=b.
\]

For this second-order Phototaxis system:

\[
\ddot X = f(V)+f_\phi(X).
\]

The fitted coefficient vector is partitioned as

\[
\theta = (\theta_\phi,\theta_f).
\]

In [22]:
# FIT_CASE_NAME = TRAIN_FIT_DATASET_NAME
# PARALLEL_JOBS =1

# fit_result_path = fit_artifact_dir / "fit_result.pkl"
# normal_system_path = fit_artifact_dir / "normal_system.npz"
# coefficients_path = fit_artifact_dir / "coefficients.npz"
# diagnostics_path = fit_artifact_dir / "diagnostics.json"
# metadata_path = fit_artifact_dir / "metadata.json"

# print("Fit artifact directory:")
# print(pretty_path(fit_artifact_dir))

In [23]:
FIT_CASE_NAME = TRAIN_FIT_DATASET_NAME
PARALLEL_JOBS = 1

print("Fit artifact directories:")
for algorithm in ("nonparametric_f", "parametric_f"):
    fit_artifact_dir = fit_repo.case_dir(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        algorithm=algorithm,
        fit_case_name=FIT_CASE_NAME,
    )
    print(algorithm, ":", pretty_path(fit_artifact_dir))

Fit artifact directories:
nonparametric_f : artifacts/fits/spp/spp_swarm_spread/nonparametric_f/dataset_train_fit_trial_001
parametric_f : artifacts/fits/spp/spp_swarm_spread/parametric_f/dataset_train_fit_trial_001


In [24]:
FIT_RESULTS = {}

# Define solver
solver = LinearSystemSolver(
    method="lstsq",
    ridge_lambda=1e-8,
    svd_tol=1e-12,
)

for ALGORITHM in ("nonparametric_f", "parametric_f"):
    # Fit or load from repository
    case_dir = fit_repo.case_dir(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        algorithm=ALGORITHM,
        fit_case_name=FIT_CASE_NAME,
    )

    fit_result_path = case_dir / "fit_result.pkl"

    if fit_result_path.exists():
        print(f"Loading existing fit_result from repository: {ALGORITHM}")

        fit_result = fit_repo.load_fit_result(
            system_name=SYSTEM_NAME,
            profile_name=PROFILE_NAME,
            algorithm=ALGORITHM,
            fit_case_name=FIT_CASE_NAME,
        )


    else:
        print(f"Fit artifact not found. Running algorithm: {ALGORITHM}")

        interaction_type = features.metadata.get(
            "interaction_type",
            train_fit_dataset.metadata.get("interaction_type"),
        )

        assert interaction_type == INTERACTION_TYPE

        assembler = NormalEquationAssembler(
            parallel_jobs=1,
            interaction_type=interaction_type,
            algorithm=ALGORITHM,
        )

        fit_result = assembler.fit(
            features=features,
            basis_bundle=basis_bundle,
            solver=solver,
            system_name=SYSTEM_NAME,
        )

        diagnostics_dict = fit_repo._diagnostics_to_dict(fit_result.diagnostics)

        fit_metadata = {
            **fit_result.metadata,

            # Explicit experiment provenance.
            "system_name": SYSTEM_NAME,
            "profile_name": PROFILE_NAME,
            "algorithm": ALGORITHM,
            "artifact_type": "normal_system_fit",
            "fit_case_name": FIT_CASE_NAME,

            # Upstream artifacts.
            "train_fit_dataset": TRAIN_FIT_DATASET_NAME,
            "feature_name": FEATURE_NAME,
            "basis_name": BASIS_NAME,

            # Fitting-window dimensions.
            "M": int(train_fit_dataset.metadata["M"]),
            "L": int(train_fit_dataset.metadata["L"]),
            "T": float(train_fit_dataset.metadata["T"]),

            "coefficient_names": ["theta", "theta_phi", "theta_f"],
        }

        fit_repo.save(
            system_name=SYSTEM_NAME,
            profile_name=PROFILE_NAME,
            fit_case_name=FIT_CASE_NAME,
            A=fit_result.A,
            b=fit_result.b,
            theta=fit_result.theta,
            n_radial=fit_result.n_radial,
            n_environment=fit_result.n_environment,
            diagnostics=diagnostics_dict,
            metadata=fit_metadata,
            overwrite=True,
        )

        fit_repo.save_fit_result(
            system_name=SYSTEM_NAME,
            profile_name=PROFILE_NAME,
            algorithm=ALGORITHM,
            fit_case_name=FIT_CASE_NAME,
            fit_result=fit_result,
            overwrite=True,
        )

        print(f"Saved fit artifacts using NormalSystemRepository: {ALGORITHM}")
    FIT_RESULTS[ALGORITHM] = fit_result



Loading existing fit_result from repository: nonparametric_f
Loading existing fit_result from repository: parametric_f


In [ ]:
# # Load saved normal system for inspection

# normal_system = fit_repo.load(
#     system_name=SYSTEM_NAME,
#     profile_name=PROFILE_NAME,
#     fit_case_name=FIT_CASE_NAME,
# )

# print("Loaded normal system keys:")
# print(normal_system.keys())

# print("metadata system_order:", normal_system["metadata"].get("system_order"))
# print("metadata environment_variable:", normal_system["metadata"].get("environment_variable"))
# print("theta:", normal_system["theta"].shape)
# print("theta_phi:", normal_system["theta_phi"].shape)
# print("theta_f:", normal_system["theta_f"].shape)

In [25]:
for ALGORITHM, fit_result in FIT_RESULTS.items():
    diag = fit_result.diagnostics

    print("\nFit diagnostics")
    print("---------------")
    print("algorithm:            ", ALGORITHM)
    print("system_name:          ", fit_result.metadata.get("system_name"))
    print("system_order:         ", fit_result.metadata.get("system_order"))
    print("model_family:         ", fit_result.metadata.get("model_family"))
    print("model_form:           ", fit_result.metadata.get("model_form"))
    print("interaction_type:     ", fit_result.metadata.get("interaction_type"))
    print("environment_variable: ", fit_result.metadata.get("environment_variable"))
    print("f_representation:     ", fit_result.metadata.get("f_representation"))

    if ALGORITHM == "parametric_f":
        print("parametric f terms:   ", fit_result.metadata.get("parametric_f_terms"))
        print("theta_f parametric:        ", fit_result.theta_f)

    print("theta shape:          ", fit_result.theta.shape)
    print("theta_phi shape:      ", fit_result.theta_phi.shape)
    print("theta_f shape:        ", fit_result.theta_f.shape)
    
    print("n_radial:             ", fit_result.n_radial)
    print("n_environment:        ", fit_result.n_environment)

    if hasattr(fit_result, "n_velocity"):
        print("n_velocity alias:     ", fit_result.n_velocity)

    print("A shape:              ", None if fit_result.A is None else fit_result.A.shape)
    print("b shape:              ", None if fit_result.b is None else fit_result.b.shape)
    print("condition number:     ", diag.condition_number)
    print("rank:                 ", diag.rank)
    print("min singular:         ", diag.min_singular)
    print("max singular:         ", diag.max_singular)
    print("solver method:        ", diag.method)

    assert fit_result.theta.shape[0] == fit_result.n_radial + fit_result.n_environment
    assert fit_result.theta_phi.shape[0] == fit_result.n_radial
    assert fit_result.theta_f.shape[0] == fit_result.n_environment

    assert fit_result.metadata.get("system_name") == SYSTEM_NAME
    assert fit_result.metadata.get("system_order") == SYSTEM_ORDER
    assert fit_result.metadata.get("interaction_type") == INTERACTION_TYPE

    expected_env_variable = "position" if SYSTEM_ORDER == 1 else "velocity"
    assert fit_result.metadata.get("environment_variable") == expected_env_variable

    assert fit_result.metadata.get("algorithm") == ALGORITHM

print("Fit consistency checks passed for all algorithms.")


Fit diagnostics
---------------
algorithm:             nonparametric_f
system_name:           spp
system_order:          2
model_family:          second_order_nonparametric_f
model_form:            ddot_X = f(V) + f_phi_E(X)
interaction_type:      energy
environment_variable:  velocity
f_representation:      nonparametric
theta shape:           (210, 1)
theta_phi shape:       (10, 1)
theta_f shape:         (200, 1)
n_radial:              10
n_environment:         200
A shape:               (210, 210)
b shape:               (210, 1)
condition number:      898882.4062383227
rank:                  210
min singular:          3.706049469192153e-06
max singular:          3.3313026645057016
solver method:         lstsq

Fit diagnostics
---------------
algorithm:             parametric_f
system_name:           spp
system_order:          2
model_family:          second_order_parametric_f
model_form:            ddot_X = f(V) + f_phi_E(X)
interaction_type:      energy
environment_variable:  velo

# Stage 4: Learned model reconstruction

Cell — Load fit_result and build learned model

Cell — Inspect learned components and smoke-test learned dynamics


Build the learned second-order SPP model from the saved fit result.

For this SPP energy-interaction model:

\[
\ddot X = \hat f(V) + \hat f_\phi(X).
\]

The learned model should expose:

- `learned_kernel`
- `learned_environment`
- `acceleration(X, V)`
- `rhs(t, state)`

In [ ]:
LEARNED_MODELS = {}
for algorithm, fit_result in FIT_RESULTS.items():

    # Load or use existing fit_result

    fit_result = fit_repo.load_fit_result(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        fit_case_name=FIT_CASE_NAME,
        algorithm=algorithm,
    )

    print("Loaded fit_result.")

    learned_model = LearnedModelFactory().build_from_fit_result(
        fit_result=fit_result,
        N=int(train_fit_dataset.metadata["N"]),
        D=int(train_fit_dataset.metadata["D"]),
    )
    LEARNED_MODELS[algorithm] = learned_model

    print("Learned model built.")
    print("system_order:", learned_model.system_order)
    print("interaction_type:", learned_model.interaction_type)
    print("N:", learned_model.N)
    print("D:", learned_model.D)
    print("state_dim:", learned_model.state_dim)
    print("algorithm :", learned_model.algorithm)

    assert learned_model.system_order == SYSTEM_ORDER
    assert learned_model.interaction_type == INTERACTION_TYPE

In [ ]:
for ALGORITHM in ("nonparametric_f", "parametric_f"):
    learned_model = LEARNED_MODELS[ALGORITHM]
    fit_result = FIT_RESULTS[ALGORITHM]

    assert fit_result.metadata["algorithm"] == ALGORITHM
    assert learned_model.algorithm == ALGORITHM

    learned_kernel = learned_model.learned_kernel
    learned_environment = learned_model.learned_environment

    print("\nAlgorithm:", ALGORITHM)
    print("Learned model algorithm:", learned_model.algorithm)
    print("Learned kernel object:", type(learned_kernel))
    print("Learned environment object:", type(learned_environment))
    print("theta_phi shape:", fit_result.theta_phi.shape)
    print("theta_f shape:", fit_result.theta_f.shape)

    X0 = train_fit_dataset.X[0, 0]
    print("X0:", X0.shape)

    if SYSTEM_ORDER == 1:
        lhs_hat = learned_model.lhs(X0)

        print("lhs_hat:", lhs_hat.shape)

        assert train_fit_dataset.V is None
        assert lhs_hat.shape == X0.shape

        print("Learned first-order lhs check passed.")

    elif SYSTEM_ORDER == 2:
        V0 = train_fit_dataset.V[0, 0]
        acc_hat = learned_model.acceleration(X0, V0)

        print("V0:", V0.shape)
        print("acc_hat:", acc_hat.shape)

        assert acc_hat.shape == X0.shape

        print("Learned second-order acceleration check passed.")

    else:
        raise ValueError(f"Unsupported SYSTEM_ORDER={SYSTEM_ORDER}")

In [ ]:
# Animation learned models for ANIMATION_N
LEARNED_MODELS_ANI = {}

for algorithm, fit_result in FIT_RESULTS.items():
    fit_result = fit_repo.load_fit_result(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        fit_case_name=FIT_CASE_NAME,
        algorithm=algorithm,
    )

    learned_model = LearnedModelFactory().build_from_fit_result(
        fit_result=fit_result,
        N=int(ANIMATION_N),
        D=int(train_fit_dataset.metadata["D"]),
        N4_kernel_scale=int(ANIMATION_N / train_fit_dataset.metadata["N"]),
    )

    LEARNED_MODELS_ANI[algorithm] = learned_model

    print("Animation learned model:", algorithm)
    print("N:", learned_model.N)
    print("D:", learned_model.D)
    print("state_dim:", learned_model.state_dim)

# Stage 5: Rollout — reconstruction and prediction

This stage rolls out the learned model and saves trajectory artifacts.

First-order systems save:

- `X_hat`
- `V_hat=None`

Second-order systems save:

- `X_hat`
- `V_hat`

In [ ]:
ROLLOUT_STAGE = "step_5"

FIT_RECON_ROLLOUT_NAME = f"{FIT_CASE_NAME}_train_fit_reconstruction"
TRAIN_FULL_ROLLOUT_NAME = f"{FIT_CASE_NAME}_train_full_prediction"
TEST_ROLLOUT_NAME = f"{FIT_CASE_NAME}_test_prediction"
TEST_ANIMATION_ROLLOUT_NAME = (
    f"{FIT_CASE_NAME}_test_animation_T{ANIMATION_END_TIME:g}_N{ANIMATION_N}"
)

print("Rollout names:")
print(FIT_RECON_ROLLOUT_NAME)
print(TRAIN_FULL_ROLLOUT_NAME)
print(TEST_ROLLOUT_NAME)
print(TEST_ANIMATION_ROLLOUT_NAME)


In [ ]:
def run_or_load_rollout(
    *,
    learned_model,
    dataset,
    rollout_name: str,
    scenario: str,
    algorithm: str,
):
    if rollout_repo.exists_case(
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        algorithm=algorithm,
        case_name=FIT_CASE_NAME,
        scenario=scenario,
        rollout_name=rollout_name,
    ):
        print("Loading existing rollout:", algorithm, rollout_name)

        return rollout_repo.load_case(
            system_name=SYSTEM_NAME,
            profile_name=PROFILE_NAME,
            algorithm=algorithm,
            case_name=FIT_CASE_NAME,
            scenario=scenario,
            rollout_name=rollout_name,
        )

    print("Running rollout:", algorithm, rollout_name)

    artifact = LearnedSystemRollout(learned_model).simulate_many(
        U0=dataset.U0,
        t_eval=dataset.t,
        method="Radau",
        rtol=1e-6,
        atol=1e-6,
        show_progress=False,
        metadata={
            "artifact_type": "rollout",
            "stage": ROLLOUT_STAGE,
            "algorithm": algorithm,

            # Learned-model provenance.
            "system_name": SYSTEM_NAME,
            "profile_name": PROFILE_NAME,
            "fit_case_name": FIT_CASE_NAME,

            # Source-dataset provenance.
            "source_system_name": dataset.metadata.get("system_name"),
            "source_profile_name": dataset.metadata.get("profile_name"),
            "source_dataset_name": dataset.metadata.get("dataset_name"),
            "source_split": dataset.metadata.get("split"),

            "system_order": SYSTEM_ORDER,
            "environment_variable": (
                "position" if SYSTEM_ORDER == 1 else "velocity"
            ),
            "rollout_name": rollout_name,
            "scenario": scenario,
            "interaction_type": learned_model.interaction_type,
        },
    )

    rollout_repo.save_case(
        artifact=artifact,
        system_name=SYSTEM_NAME,
        profile_name=PROFILE_NAME,
        algorithm=algorithm,
        case_name=FIT_CASE_NAME,
        scenario=scenario,
        rollout_name=rollout_name,
        overwrite=True,
    )

    return artifact


def print_rollout_summary(name, artifact):
    print(name)
    print("algorithm:", artifact.metadata.get("algorithm"))
    print("t:", artifact.t.shape)
    print("X_hat:", artifact.X_hat.shape)
    print("V_hat:", None if artifact.V_hat is None else artifact.V_hat.shape)
    print("system_order:", artifact.metadata.get("system_order"))
    print("scenario:", artifact.metadata.get("scenario"))
    print("source_profile:", artifact.metadata.get("source_profile_name"))


In [ ]:
# Reconstruction rollout on train-fit window
ROLLOUT_RESULTS = {}

for ALGORITHM, learned_model in LEARNED_MODELS.items():
    fit_recon_artifact = run_or_load_rollout(
        learned_model=learned_model,
        dataset=train_fit_dataset,
        rollout_name=FIT_RECON_ROLLOUT_NAME,
        scenario="train_fit_reconstruction",
        algorithm=ALGORITHM,
    )

    ROLLOUT_RESULTS.setdefault(ALGORITHM, {})
    ROLLOUT_RESULTS[ALGORITHM]["train_fit_reconstruction"] = (
        fit_recon_artifact
    )

    print_rollout_summary(
        f"{ALGORITHM} train-fit reconstruction:",
        fit_recon_artifact,
    )


In [ ]:
# Prediction rollout on full training horizon
for ALGORITHM, learned_model in LEARNED_MODELS.items():
    train_full_artifact = run_or_load_rollout(
        learned_model=learned_model,
        dataset=train_dataset,
        rollout_name=TRAIN_FULL_ROLLOUT_NAME,
        scenario="train_full_prediction",
        algorithm=ALGORITHM,
    )

    ROLLOUT_RESULTS.setdefault(ALGORITHM, {})
    ROLLOUT_RESULTS[ALGORITHM]["train_full_prediction"] = (
        train_full_artifact
    )

    print_rollout_summary(
        f"{ALGORITHM} train-full prediction:",
        train_full_artifact,
    )


In [ ]:
# Prediction rollout on unseen test initial conditions
for ALGORITHM, learned_model in LEARNED_MODELS.items():
    test_artifact = run_or_load_rollout(
        learned_model=learned_model,
        dataset=test_dataset,
        rollout_name=TEST_ROLLOUT_NAME,
        scenario="test_prediction",
        algorithm=ALGORITHM,
    )

    ROLLOUT_RESULTS.setdefault(ALGORITHM, {})
    ROLLOUT_RESULTS[ALGORITHM]["test_prediction"] = test_artifact

    print_rollout_summary(
        f"{ALGORITHM} test prediction:",
        test_artifact,
    )


In [ ]:
# Animation dataset and prediction rollout.
#
# Generation is deferred until this stage because
# N=100, T=20, L=2001 is substantially more expensive
# than the standard learning datasets.

from dataclasses import asdict

from ips_learning.experiments.generate_dataset_collection import (
    run_one_dataset_from_profile,
)

test_animation_dir = (
    ANIMATION_DATASET_DIR
    / TEST_SPLIT
    / TEST_ANIMATION_DATASET_NAME
)

if not dataset_artifact_complete(test_animation_dir):
    print("Generating animation test dataset.")

    animation_profile = build_spp_dataset_profile(ANI_PROFILE_NAME)

    animation_split_config = {
        **asdict(animation_profile.test),
        "dataset_name": TEST_ANIMATION_DATASET_NAME,
        "dataset_role": "test_animation",
        "source_dataset_name": TEST_DATASET_NAME,
        "N": ANIMATION_N,
        "L": ANIMATION_NUM_FRAMES,
        "T": ANIMATION_END_TIME,
    }

    run_one_dataset_from_profile(
        profile=animation_profile,
        split=TEST_SPLIT,
        dataset_name=TEST_ANIMATION_DATASET_NAME,
        M=animation_profile.test.M,
        seed=animation_profile.test.seed,
        L=ANIMATION_NUM_FRAMES,
        T=ANIMATION_END_TIME,
        split_config=animation_split_config,
        N_override=ANIMATION_N,
    )

else:
    print("Animation test dataset already exists. Skipping generation.")


test_animation_dataset = dataset_repo.load(
    system_name=SYSTEM_NAME,
    profile_name=ANI_PROFILE_NAME,
    split=TEST_SPLIT,
    dataset_name=TEST_ANIMATION_DATASET_NAME,
)

print("Animation dataset loaded.")


# Prediction rollout on unseen animation initial conditions.
for ALGORITHM, learned_model in LEARNED_MODELS_ANI.items():
    animation_artifact = run_or_load_rollout(
        learned_model=learned_model,
        dataset=test_animation_dataset,
        rollout_name=TEST_ANIMATION_ROLLOUT_NAME,
        scenario="test_animation_prediction",
        algorithm=ALGORITHM,
    )

    ROLLOUT_RESULTS.setdefault(ALGORITHM, {})
    ROLLOUT_RESULTS[ALGORITHM]["test_animation_prediction"] = (
        animation_artifact
    )

    print_rollout_summary(
        f"{ALGORITHM} test animation prediction:",
        animation_artifact,
    )
